In [3]:
import pandas as pd
import tqdm
import numpy as np
import glob
from collections import defaultdict
import misc, os
import torch as th
from PIL import Image

def read_params(path):
    params = pd.read_csv(path, header=None, sep=" ", index_col=False, lineterminator='\n')
    params.rename(columns={0:'img_name'}, inplace=True)
    params = params.set_index('img_name').T.to_dict('list')
    return params

def swap_key(params):
    params_s = defaultdict(dict)
    for params_name, v in params.items():
        for img_name, params_value in v.items():
            params_s[img_name][params_name] = np.array(params_value).astype(np.float64)

    return params_s

def load_deca_params(deca_dir, cfg):
    deca_params = {}

    # face params 
    params_key = ['light']
    for k in tqdm.tqdm(params_key, desc="Loading deca params..."):
        params_path = glob.glob(f"{deca_dir}/*{k}-anno.txt")
        for path in params_path:
            deca_params[k] = read_params(path=path)
    
    deca_params = swap_key(deca_params)
    return deca_params

In [15]:
data = 'holo'
if data == 'holo':
    testing_lightfile = '/data/mint/DPM_Dataset/HoloRelighting/params/valid/ffhq-valid-light-anno.txt'
    testing_img_path = '/data/mint/DPM_Dataset/HoloRelighting/images_aligned/valid/'
    testing_sj = {'f4_main_ref_1':[], 'f6_sup_ref_1':[], 'f6_sup_ref_3':[]}
elif data == 'switch':
    testing_lightfile = '/data/mint/DPM_Dataset/SwitchLight/params/valid/ffhq-valid-light-anno.txt'
    testing_img_path = '/data/mint/DPM_Dataset/SwitchLight/images_aligned/valid/'
    testing_sj = {'fig10_main_ref_1':[], 'fig6_main_ref_3':[]}
elif data == 'switch_lim':
    testing_lightfile = '/data/mint/DPM_Dataset/SwitchLight_limitation/params/valid/ffhq-valid-light-anno.txt'
    testing_img_path = '/data/mint/DPM_Dataset/SwitchLight_limitation/images_aligned/valid/'
    testing_sj = {'lim1_dst':[]}

testing_sh = read_params(testing_lightfile)
for k in testing_sj.keys():
    sh_k = k + '.png'
    print(sh_k, testing_sh[sh_k])
    tmp_wclip, tmp_woclip = misc.drawSphere(sh=th.tensor(testing_sh[sh_k]).view(9, 3))
    tmp_wclip = tmp_wclip.permute(1, 2, 0).cpu().numpy()
    ball = (tmp_wclip * 255.0).astype(np.uint8)
    testing_sj[k] = {
        'ball':ball,
        'mask': (ball > 0) * 1.0
    }
    Image.fromarray(testing_sj[k]['ball']).save(f'./{k}_ball.png')

f4_main_ref_1.png [3.104291, 3.0836577, 3.0626488, -0.13564305, -0.10477726, -0.0863187, 0.04033911, 0.026881194, 0.019327877, -0.4341407, -0.48655725, -0.54491657, 0.055952586, 0.05372095, 0.05414563, -0.72613645, -0.70903236, -0.69931114, -0.009984996, -0.018662192, -0.025435757, 0.51946473, 0.5080597, 0.50242925, 0.88298374, 0.8731629, 0.85794556]
f6_sup_ref_1.png [3.3270822, 3.3185697, 3.303788, -0.014658509, 0.0009641759, 0.0061141774, -0.08797248, -0.09360092, -0.09798575, -0.50940645, -0.55517274, -0.59939975, 0.049810123, 0.0499696, 0.050668646, -0.25115103, -0.24026938, -0.23895824, 0.27249783, 0.27167326, 0.26712778, 0.82217276, 0.82744896, 0.82474965, 0.52345866, 0.49902317, 0.48403338]
f6_sup_ref_3.png [3.6180534, 3.6151776, 3.6047828, -0.5000785, -0.5070103, -0.50851846, -0.06321418, -0.06038925, -0.059832133, -0.53212774, -0.55907196, -0.5857616, 0.054349054, 0.055311933, 0.055494923, 0.67443645, 0.6690543, 0.6691961, 0.39126164, 0.39188227, 0.39043045, 0.99284506, 0.9994